In [5]:
import pandas as pd
import platform
import glob

In [6]:
# Check the operating system - file location is different if its Windows or OS
if platform.system() == 'Windows':
    path = "G:/My Drive/EarthEngineData"
else:
    # For Holden's Mac
    path = "/Users/holden/Personal Projects/flame-flame-fruit/FireData"

files = glob.glob(path + "/*.csv")

In [7]:
# Throw all the files into one large pandas dataframe (this took me 22m to run btw)
df_list = []
for file in files:
    # Best practice is probably to put this all in a try catch but it worked for me for now...
    df = pd.read_csv(file)
    
    # Only add some of the days with no fire, since with too many it will skew predictions (since fire is rare)
    no_fires = df[df['T21_max'] == 0].sample(frac=0.1, random_state=42) #choosing 10% - we can change this
    # Every day with fire
    fires = df[df['T21_max'] > 0]
    
    both = pd.concat([fires, no_fires])
    df_list.append(both)

#Combine all the dataframes into one
final_df = pd.concat(df_list, ignore_index=True) #ignore_index to reset the row numbers on each list
print("final dataset size: ", final_df.shape)

final dataset size:  (11271324, 30)


In [19]:
# Parse grid cell x,y indices out of system:index (format: YYYYMMDD_x,y)
# These represent which 4x4km grid cell in Colorado the row belongs to
final_df[['grid_x', 'grid_y']] = final_df['system:index'].str.split('_').str[1].str.split(',', expand=True).astype(int)

# Extracts the month number from the date (1-12) and adds it as a new column
final_df['month'] = pd.to_datetime(final_df['date']).dt.month

# Extracts year from the date and adds it as a new column
# Not necessary but could be useful to see if fire danger is increasing over time as a side project
final_df['year'] = pd.to_datetime(final_df['date']).dt.year

# Creates a 0/1 column in case there is a fire
final_df['fire'] = (final_df['T21_max'] > 0).astype(int)



In [22]:
# This just helps visualize the data frame's structure
print(final_df.columns.tolist())
final_df.head(5)

['system:index', 'T21_max', 'T21_mean', 'T21_stdDev', 'aspect_max', 'aspect_mean', 'aspect_stdDev', 'date', 'elevation_max', 'elevation_mean', 'elevation_stdDev', 'erc_max', 'erc_mean', 'erc_stdDev', 'pr_max', 'pr_mean', 'pr_stdDev', 'rmin_max', 'rmin_mean', 'rmin_stdDev', 'slope_max', 'slope_mean', 'slope_stdDev', 'tmmx_max', 'tmmx_mean', 'tmmx_stdDev', 'vs_max', 'vs_mean', 'vs_stdDev', '.geo', 'grid_x', 'grid_y', 'month', 'year', 'fire']


,system:index,T21_max,T21_mean,T21_stdDev,aspect_max,aspect_mean,aspect_stdDev,date,elevation_max,elevation_mean,...,tmmx_stdDev,vs_max,vs_mean,vs_stdDev,.geo,grid_x,grid_y,month,year,fire
0,"20090106_119,1109",323.500000,217.290663,120.561332,109.0,103.590361,8.552128,2009-01-06,1719,1634.042169,...,0.890324,11.414818,9.866028,0.766249,"{""type"":""MultiPoint"",""coordinates"":[]}",119,1109,1,2009,1
1,"20090106_119,1110",323.500000,31.087087,120.561332,111.0,97.243243,9.154537,2009-01-06,1751,1667.360360,...,1.030267,10.506145,9.062752,0.698769,"{""type"":""MultiPoint"",""coordinates"":[]}",119,1110,1,2009,1
2,"20090112_137,1057",317.700012,31.492129,118.399804,46.0,26.851312,9.231588,2009-01-12,1458,1427.527697,...,0.055919,3.610772,3.552354,0.034750,"{""type"":""MultiPoint"",""coordinates"":[]}",137,1057,1,2009,1
3,"20090112_138,1057",317.700012,122.948921,137.568141,76.0,32.306502,22.015619,2009-01-12,1449,1421.492260,...,0.050070,3.635362,3.595614,0.024742,"{""type"":""MultiPoint"",""coordinates"":[]}",138,1057,1,2009,1
4,"20090112_137,1058",317.700012,19.856251,118.399804,102.0,75.860119,32.361242,2009-01-12,1434,1401.470238,...,0.137627,3.566329,3.494493,0.041712,"{""type"":""MultiPoint"",""coordinates"":[]}",137,1058,1,2009,1
